# Heart Disease Dataset - Preprocessing Notebook

This notebook explains the preprocessing steps applied to the Heart Disease dataset before training the machine learning models.

In [8]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import os

PROJECT_ROOT = os.path.abspath('.')
data_path = os.path.join(PROJECT_ROOT, 'data', 'heartt_cleveland_cleaned.csv')
df = pd.read_csv(data_path)
print(f'Dataset shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()

Dataset shape: (303, 14)
Columns: ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,1
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0


## Step 1: Check for Missing Values

First, we examine the dataset to identify any missing values (NaN) in our features.

In [9]:
# Check for missing values
print('Missing values per column:')
missing = df.isnull().sum()
print(missing[missing > 0])
print(f'\nTotal rows with missing values: {df.isnull().any(axis=1).sum()}')
print(f'Total missing values: {df.isnull().sum().sum()}')

Missing values per column:
ca      4
thal    2
dtype: int64

Total rows with missing values: 6
Total missing values: 6


## Step 2: Why Not Drop Missing Values?

**Problem with dropping (df.dropna()):**
- Loses valuable data that could improve model training
- Reduces dataset size, potentially hurting model performance
- May introduce bias if missing values are not random

**Solution: Mean Imputation**
- Replace missing values with the mean (average) of that column
- Keeps all rows in the dataset
- Simple and effective for numerical data

In [11]:
# Separate features and target
X = df.drop('target', axis=1)
y = df['target']

print('Feature columns:')
print(X.columns.tolist())
print(f'\nNumber of features: {X.shape[1]}')
print(f'Number of samples before imputation: {X.shape[0]}')

Feature columns:
['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']

Number of features: 13
Number of samples before imputation: 303


## Step 3: Apply Mean Imputation

We use Scikit-learn's `SimpleImputer` with strategy='mean' to fill missing values.

In [12]:
# Create imputer that fills missing values with mean
imputer = SimpleImputer(strategy='mean')

# Fit imputer on training data and transform
X_imputed = pd.DataFrame(
    imputer.fit_transform(X),
    columns=X.columns
)

print('After imputation:')
print(f'Missing values: {X_imputed.isnull().sum().sum()}')
print(f'Number of samples: {X_imputed.shape[0]}')
X_imputed.head()

After imputation:
Missing values: 0
Number of samples: 303


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0


## Step 4: Feature Scaling (Standardization)

We scale features to have zero mean and unit variance using `StandardScaler`.

**Why scale?**
- Many ML algorithms (SVM, Logistic Regression) perform better with scaled data
- Ensures all features contribute equally regardless of their units
- Helps gradient-based optimization converge faster

In [13]:
# Scale the features
scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X_imputed),
    columns=X_imputed.columns
)

print('After scaling:')
print(f'Mean of each feature: ~0 (actual: {X_scaled.mean().mean():.6f})')
print(f'Std of each feature: ~1 (actual: {X_scaled.std().mean():.6f})')
X_scaled.head()

After scaling:
Mean of each feature: ~0 (actual: 0.000000)
Std of each feature: ~1 (actual: 1.001654)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
0,0.948726,0.686202,-2.251775,0.757525,-0.264900,2.394438,1.016684,0.017197,-0.696631,1.087338,2.274579,-0.723095,0.655818
1,1.392002,0.686202,0.877985,1.611220,0.760415,-0.417635,1.016684,-1.821905,1.435481,0.397182,0.649113,2.503851,-0.898522
2,1.392002,0.686202,0.877985,-0.665300,-0.342283,-0.417635,1.016684,-0.902354,1.435481,1.346147,0.649113,1.428203,1.173931
3,-1.932564,0.686202,-0.165268,-0.096170,0.063974,-0.417635,-0.996749,1.637359,-0.696631,2.122573,2.274579,-0.723095,-0.898522
4,-1.489288,-1.457296,-1.208521,-0.096170,-0.825922,-0.417635,1.016684,0.980537,-0.696631,0.310912,-0.976352,-0.723095,-0.898522


## Step 5: Save Preprocessing Objects

We save the imputer and scaler so they can be used during prediction on new data.

In [14]:
import joblib

models_dir = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(models_dir, exist_ok=True)

# Save the imputer and scaler
joblib.dump(imputer, os.path.join(models_dir, 'imputer.pkl'))
joblib.dump(scaler, os.path.join(models_dir, 'scaler.pkl'))
joblib.dump(X.columns.tolist(), os.path.join(models_dir, 'feature_names.pkl'))

print('Preprocessing objects saved:')
print('- imputer.pkl: Fills missing values with mean')
print('- scaler.pkl: Standardizes features')
print('- feature_names.pkl: List of feature names')

Preprocessing objects saved:
- imputer.pkl: Fills missing values with mean
- scaler.pkl: Standardizes features
- feature_names.pkl: List of feature names


## Summary

| Step | Action | Before | After |
|------|--------|--------|-------|
| 1 | Load Data | Raw CSV | DataFrame |
| 2 | Check Missing | May have NaN | Identify locations |
| 3 | Imputation | dropna() lost rows | Mean fill keeps all rows |
| 4 | Scaling | Different ranges | Zero mean, unit variance |
| 5 | Save | - | Reusable .pkl files |